# Phase 4 — Two-Stage Tree Health Classification

## Architecture
```
Satellite tile
      ↓
[Stage 1] DeepForest tree detector (1-class)
      → bounding boxes of all trees
      ↓
[Stage 2] ResNet-18 crop classifier (Healthy / Seca)
      → per-tree label + confidence
      ↓
Colour-coded map: green = Healthy, red = Seca
```

## Why this is better than 1-stage
- Stage 1 uses the pretrained backbone already good at detecting tree shapes
- Stage 2 sees only the tree crop — no background noise, much cleaner signal
- 892 Seca crops is enough for a classifier (vs too few for a detector)
- Models are independent and improvable separately

In [ ]:
# ── Cell 0: Setup ──────────────────────────────────────────────────────────
import subprocess, sys
def pip(pkg): subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

import platform
print(f'Platform  : {platform.system()} {platform.machine()}')

import torch
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ── Cell 1: Imports & Config ───────────────────────────────────────────────
import os, json, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

# ── Seeds ──────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT       = Path.home() / 'quercus_v2'
DATA_DIR   = ROOT / 'data'
CROPS_DIR  = DATA_DIR / 'crops'
MODELS_DIR = ROOT / 'models'
REPORTS    = ROOT / 'reports'
for d in [CROPS_DIR / 'train' / 'Healthy', CROPS_DIR / 'train' / 'Seca',
          CROPS_DIR / 'val'   / 'Healthy', CROPS_DIR / 'val'   / 'Seca',
          CROPS_DIR / 'test'  / 'Healthy', CROPS_DIR / 'test'  / 'Seca',
          MODELS_DIR, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

# Source data from Phase 3 run
SRC_DATA = Path.home() / 'quercus_train' / 'data'
TRAIN_CSV = SRC_DATA / 'train.csv'
VAL_CSV   = SRC_DATA / 'val.csv'
TEST_CSV  = SRC_DATA / 'test.csv'

# ── Hyperparameters ────────────────────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
CROP_SIZE   = 96          # px — resize all crops to this
BATCH_SIZE  = 64 if DEVICE == 'cuda' else 16
NUM_EPOCHS  = 30
LR          = 3e-4
NUM_WORKERS = 4 if DEVICE == 'cuda' else 0
PADDING     = 8           # px padding around each annotation box when cropping

print(f'Device    : {DEVICE}')
print(f'Crop size : {CROP_SIZE}px  Padding: {PADDING}px')
print(f'Epochs    : {NUM_EPOCHS}  LR: {LR}  Batch: {BATCH_SIZE}')

## Cell 2: Extract Tree Crops from Annotations

For each annotated bounding box in train/val/test CSVs, we crop the corresponding
image region and save it as `crops/<split>/<Healthy|Seca>/<uid>.jpg`.
A small padding around the box is added to give the classifier some context.

In [ ]:
# ── Cell 2: Extract crops ──────────────────────────────────────────────────

def extract_crops(csv_path, split_name, padding=PADDING):
    df = pd.read_csv(csv_path)
    counts = {'Healthy': 0, 'Seca': 0, 'skip': 0}
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f'{split_name}'):
        label = str(row['label']).strip()
        if label not in ('Healthy', 'Seca'):
            counts['skip'] += 1
            continue
        
        img_path = Path(row['image_path'])
        if not img_path.is_absolute():
            img_path = SRC_DATA / split_name / img_path.name
        if not img_path.exists():
            # try locating by filename in source data tree
            candidates = list(SRC_DATA.rglob(img_path.name))
            if not candidates:
                counts['skip'] += 1
                continue
            img_path = candidates[0]
        
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            counts['skip'] += 1
            continue
        
        W, H = img.size
        x1 = max(0, int(row['xmin']) - padding)
        y1 = max(0, int(row['ymin']) - padding)
        x2 = min(W, int(row['xmax']) + padding)
        y2 = min(H, int(row['ymax']) + padding)
        
        if x2 <= x1 or y2 <= y1:
            counts['skip'] += 1
            continue
        
        crop = img.crop((x1, y1, x2, y2))
        out_dir = CROPS_DIR / split_name / label
        out_path = out_dir / f'{split_name}_{idx:06d}.jpg'
        crop.save(out_path, quality=95)
        counts[label] += 1
    
    return counts

print('Extracting crops from annotations...')
stats = {}
for split, csv in [('train', TRAIN_CSV), ('val', VAL_CSV), ('test', TEST_CSV)]:
    stats[split] = extract_crops(csv, split)

print()
print(f"{'Split':<8} {'Healthy':>10} {'Seca':>8} {'Skip':>6}")
print('-' * 36)
for split, c in stats.items():
    print(f"{split:<8} {c['Healthy']:>10} {c['Seca']:>8} {c['skip']:>6}")

total_seca = sum(s['Seca'] for s in stats.values())
total_healthy = sum(s['Healthy'] for s in stats.values())
print(f'\nTotal crops: {total_healthy + total_seca}')
print(f'Class ratio: Healthy {total_healthy/(total_healthy+total_seca)*100:.1f}% / Seca {total_seca/(total_healthy+total_seca)*100:.1f}%')

## Cell 3: Visualise Sample Crops

Sanity check — show a grid of Healthy and Seca crops to confirm extraction worked.

In [ ]:
# ── Cell 3: Visualise sample crops ────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.suptitle('Sample Crops — Healthy (top) vs Seca (bottom)', fontsize=13, fontweight='bold')

for row_i, label in enumerate(['Healthy', 'Seca']):
    crops = sorted((CROPS_DIR / 'train' / label).glob('*.jpg'))[:8]
    for col_i, cp in enumerate(crops):
        ax = axes[row_i, col_i]
        ax.imshow(Image.open(cp))
        ax.axis('off')
        if col_i == 0:
            ax.set_ylabel(label, fontsize=11, color='green' if label=='Healthy' else 'red',
                          fontweight='bold', rotation=90, labelpad=4)

plt.tight_layout()
plt.savefig(REPORTS / 'sample_crops.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/sample_crops.png')

## Cell 4: Stage 2 — Dataset & DataLoaders

- Training uses `WeightedRandomSampler` to balance Healthy/Seca batches (key fix vs Phase 3)
- Augmentation: horizontal/vertical flip, color jitter, random rotation (train only)
- Validation and test: only resize + normalize

In [ ]:
# ── Cell 4: Dataset & DataLoaders ─────────────────────────────────────────
from torchvision.datasets import ImageFolder

MEAN = [0.485, 0.456, 0.406]   # ImageNet stats
STD  = [0.229, 0.224, 0.225]

train_tfm = T.Compose([
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tfm = T.Compose([
    T.Resize((CROP_SIZE, CROP_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

train_ds = ImageFolder(str(CROPS_DIR / 'train'), transform=train_tfm)
val_ds   = ImageFolder(str(CROPS_DIR / 'val'),   transform=eval_tfm)
test_ds  = ImageFolder(str(CROPS_DIR / 'test'),  transform=eval_tfm)

CLASS_NAMES = train_ds.classes   # ['Healthy', 'Seca'] alphabetically
print(f'Classes : {CLASS_NAMES}')
print(f'Train   : {len(train_ds)} crops')
print(f'Val     : {len(val_ds)} crops')
print(f'Test    : {len(test_ds)} crops')

# ── Weighted sampler to fix class imbalance ────────────────────────────────
targets = torch.tensor([t for _, t in train_ds.samples])
class_counts = torch.bincount(targets).float()
class_weights = 1.0 / class_counts
sample_weights = class_weights[targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'))

print(f'\nClass counts (train): Healthy={int(class_counts[0])}, Seca={int(class_counts[1])}')
print(f'WeightedSampler active — each batch will be ~50/50 Healthy/Seca')

## Cell 5: Stage 2 Model — ResNet-18 Binary Classifier

We replace the final FC layer with a 2-class head. The backbone (pre-trained on ImageNet)
is kept frozen for the first 5 epochs (warmup), then unfrozen for the remaining epochs.
This is the same Domain Adaptation principle as Stage 1: preserve prior knowledge, adapt gently.

In [ ]:
# ── Cell 5: Build ResNet-18 classifier ────────────────────────────────────
def build_classifier(freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace final FC: 512 → 256 → 2
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 2)
    )
    return model.to(DEVICE)

classifier = build_classifier(freeze_backbone=True)
trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total     = sum(p.numel() for p in classifier.parameters())
print(f'ResNet-18 loaded — {total/1e6:.1f}M params total, {trainable/1e3:.0f}K trainable (head only)')

# ── Loss: weighted CrossEntropy to further penalise Seca errors ───────────
# class_counts[0]=Healthy, class_counts[1]=Seca
loss_weights = (class_counts.max() / class_counts).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=loss_weights)
print(f'Loss weights: Healthy={loss_weights[0]:.3f}, Seca={loss_weights[1]:.3f}')

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, classifier.parameters()),
                        lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## Cell 6: Train the Classifier

Training loop with:
- Warmup (epochs 0-4): backbone frozen, only head trained
- Fine-tune (epoch 5+): full network unfrozen with lower LR
- Early stopping on val F1-Seca (patience=8)
- Best model saved by F1-Seca (not accuracy, to avoid majority-class bias)

In [ ]:
# ── Cell 6: Training loop ──────────────────────────────────────────────────
WARMUP_EPOCHS   = 5
PATIENCE        = 8
CLASSIFIER_PATH = MODELS_DIR / 'stage2_classifier.pt'

train_losses, val_losses, val_f1_seca, val_f1_overall = [], [], [], []
best_f1_seca = 0.0
patience_counter = 0

def run_epoch(loader, train=False):
    classifier.train() if train else classifier.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            logits = classifier(imgs)
            loss   = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    avg_loss = total_loss / len(loader.dataset)
    f1_seca  = f1_score(all_labels, all_preds, labels=[1], average='macro', zero_division=0)
    f1_ovr   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return avg_loss, f1_seca, f1_ovr

print(f'Training {NUM_EPOCHS} epochs (warmup={WARMUP_EPOCHS}, patience={PATIENCE})')
print(f'{"Epoch":<6} {"TrainLoss":>10} {"ValLoss":>9} {"F1-Seca":>9} {"F1-Ovr":>8} {"LR":>9}')
print('-' * 58)

for epoch in range(NUM_EPOCHS):
    # Unfreeze backbone after warmup
    if epoch == WARMUP_EPOCHS:
        for param in classifier.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(classifier.parameters(), lr=LR * 0.1, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS)
        print(f'  → Backbone unfrozen at epoch {epoch}, LR reduced to {LR*0.1}')
    
    tr_loss, _, _      = run_epoch(train_loader, train=True)
    vl_loss, f1s, f1o  = run_epoch(val_loader,   train=False)
    scheduler.step()
    
    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    val_f1_seca.append(f1s)
    val_f1_overall.append(f1o)
    
    lr_now = optimizer.param_groups[0]['lr']
    marker = ''
    if f1s > best_f1_seca:
        best_f1_seca = f1s
        patience_counter = 0
        torch.save(classifier.state_dict(), CLASSIFIER_PATH)
        marker = '  ← best'
    else:
        patience_counter += 1
    
    print(f'{epoch:<6} {tr_loss:>10.4f} {vl_loss:>9.4f} {f1s:>9.4f} {f1o:>8.4f} {lr_now:>9.2e}{marker}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (patience={PATIENCE})')
        break

print(f'\nBest F1-Seca (val): {best_f1_seca:.4f}')
print(f'Model saved: {CLASSIFIER_PATH}')

## Cell 7: Training Curves

In [ ]:
# ── Cell 7: Training curves ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

epochs_ran = range(len(train_losses))

ax1.plot(epochs_ran, train_losses, label='Train loss', color='steelblue')
ax1.plot(epochs_ran, val_losses,   label='Val loss',   color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Stage 2 — Loss curves'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, val_f1_seca,    label='F1 Seca',    color='crimson', linewidth=2)
ax2.plot(epochs_ran, val_f1_overall, label='F1 Overall', color='steelblue')
ax2.axhline(0.0,  color='gray', linestyle='--', alpha=0.5, label='Phase3 F1-Seca baseline')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1')
ax2.set_title('Stage 2 — F1 curves'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(REPORTS / 'stage2_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/stage2_training_curves.png')

## Cell 8: Evaluate Classifier on Test Crops

Load best checkpoint → run on test set → classification report + confusion matrix.

In [ ]:
# ── Cell 8: Test evaluation (Stage 2 classifier only) ─────────────────────
classifier.load_state_dict(torch.load(CLASSIFIER_PATH, map_location=DEVICE))
classifier.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        logits = classifier(imgs)
        probs  = torch.softmax(logits, dim=1)
        preds  = logits.argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())  # P(Seca)

report = classification_report(all_labels, all_preds,
                                target_names=CLASS_NAMES, zero_division=0)
print('=== Stage 2 Classifier — Test Set Results ===')
print(report)

f1_seca_test    = f1_score(all_labels, all_preds, labels=[1], average='macro', zero_division=0)
f1_healthy_test = f1_score(all_labels, all_preds, labels=[0], average='macro', zero_division=0)
f1_overall_test = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f'F1 Overall  : {f1_overall_test:.4f}')
print(f'F1 Healthy  : {f1_healthy_test:.4f}')
print(f'F1 Seca     : {f1_seca_test:.4f}  ← key metric')

In [ ]:
# ── Cell 9: Confusion matrix ───────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True',      fontsize=12)
ax.set_title(f'Stage 2 Classifier — Confusion Matrix\n'
             f'F1-Seca={f1_seca_test:.3f}  F1-Overall={f1_overall_test:.3f}', fontsize=11)
plt.tight_layout()
plt.savefig(REPORTS / 'stage2_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/stage2_confusion_matrix.png')

## Cell 10: Full Two-Stage Pipeline on Test Images

- **Stage 1**: Phase 3 fine-tuned DeepForest (2-class, but used purely as a tree locator — we ignore its class labels and re-classify with Stage 2)
- For each detected box: crop → Stage 2 ResNet-18 classifier → final label
- Visualize with green/red bounding boxes vs white-dashed GT

Using the fine-tuned model for Stage 1 is better than the generic baseline because it was trained on Dehesa imagery — it knows the scale, color, and texture of Spanish holm oaks.

In [ ]:
# ── Cell 10: Full pipeline ─────────────────────────────────────────────────
from deepforest import main as df_main

# Load Stage 1: try fine-tuned model via Lightning's native load_from_checkpoint,
# which correctly reconstructs the 2-class architecture before loading weights.
# Fallback to pretrained baseline if it fails (baseline is good enough for tree localization).
STAGE1_PATH = Path.home() / 'quercus_train' / 'models' / 'deepforest_dehesa_finetuned.pt'
print('Loading Stage 1 detector...')
try:
    detector = df_main.deepforest.load_from_checkpoint(str(STAGE1_PATH))
    detector.label_dict = {'Healthy': 0, 'Seca': 1}
    print(f'Stage 1: fine-tuned model loaded via load_from_checkpoint')
except Exception as e:
    print(f'Stage 1: fine-tuned load failed ({e.__class__.__name__}: {str(e)[:80]})')
    print('Stage 1: falling back to pretrained baseline (weecology/deepforest-tree)')
    detector = df_main.deepforest()
    detector.load_model()
detector.config.score_thresh = 0.15
detector.eval()
print('Stage 1 ready.')

# Load Stage 2 best checkpoint
classifier.load_state_dict(torch.load(CLASSIFIER_PATH, map_location=DEVICE))
classifier.eval()

def run_pipeline(img_path):
    """
    Stage 1: detect all trees. Stage 2: classify each crop as Healthy or Seca.
    Returns: list of (x1, y1, x2, y2, label, confidence) tuples.
    """
    try:
        preds_df = detector.predict_image(path=str(img_path))
    except Exception as e:
        print(f'  Stage 1 error on {Path(img_path).name}: {e}')
        return []
    if preds_df is None or len(preds_df) == 0:
        return []

    img = Image.open(img_path).convert('RGB')
    results = []
    for _, row in preds_df.iterrows():
        x1 = max(0, int(row['xmin']))
        y1 = max(0, int(row['ymin']))
        x2 = min(img.width,  int(row['xmax']))
        y2 = min(img.height, int(row['ymax']))
        crop = img.crop((max(0, x1-PADDING), max(0, y1-PADDING),
                         min(img.width, x2+PADDING), min(img.height, y2+PADDING)))
        tensor = eval_tfm(crop).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = classifier(tensor)
            probs  = torch.softmax(logits, dim=1)[0]
            pred   = logits.argmax(1).item()
        results.append((x1, y1, x2, y2, CLASS_NAMES[pred], float(probs[pred])))
    return results


def visualise_pipeline(img_path, results, gt_df=None, title=''):
    img_arr = np.array(Image.open(img_path).convert('RGB'))
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img_arr)
    colours = {'Healthy': '#00cc44', 'Seca': '#ff3333'}
    for (x1, y1, x2, y2, label, conf) in results:
        c = colours.get(label, 'orange')
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=c, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, max(0, y1-3), f'{label[:1]} {conf:.2f}',
                color=c, fontsize=6, fontweight='bold',
                bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))
    if gt_df is not None:
        fname = Path(img_path).name
        gt = gt_df[gt_df['image_path'].str.contains(fname, na=False)]
        for _, row in gt.iterrows():
            rect = patches.Rectangle((row['xmin'], row['ymin']),
                                       row['xmax']-row['xmin'], row['ymax']-row['ymin'],
                                       linewidth=1.5, edgecolor='white',
                                       facecolor='none', linestyle='--')
            ax.add_patch(rect)
    healthy_n = sum(1 for r in results if r[4] == 'Healthy')
    seca_n    = sum(1 for r in results if r[4] == 'Seca')
    ax.set_title(f'{title}  |  Predicted: {healthy_n} Healthy (green), {seca_n} Seca (red)\n'
                 f'White dashed = Ground Truth boxes', fontsize=10)
    ax.axis('off')
    return fig


# ── Run on test images ─────────────────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
test_imgs = sorted({Path(r['image_path']).name for _, r in test_df.iterrows()})
test_img_paths = []
for name in test_imgs[:5]:
    candidates = list(SRC_DATA.rglob(name))
    if candidates:
        test_img_paths.append(candidates[0])

print(f'\nRunning full pipeline on {len(test_img_paths)} test images...')
pipeline_results = {}
for p in test_img_paths:
    res = run_pipeline(p)
    pipeline_results[p] = res
    h = sum(1 for r in res if r[4] == 'Healthy')
    s = sum(1 for r in res if r[4] == 'Seca')
    print(f'  {p.name}: {len(res)} trees detected → {h} Healthy, {s} Seca')

In [ ]:
# ── Cell 11: Save pipeline visualisations ─────────────────────────────────
for i, (img_path, results) in enumerate(list(pipeline_results.items())[:3], 1):
    fig = visualise_pipeline(img_path, results, gt_df=test_df, title=f'Test tile {i}')
    out = REPORTS / f'pipeline_result_{i}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## Cell 12: Compare Stage 1+2 vs Phase 3 (1-stage)

We compare the two approaches on the same GT annotations:
- Phase 3 (1-stage): F1=0.669 overall, F1-Seca=0.000
- Phase 4 (2-stage): this run

In [ ]:
# ── Cell 12: Comparison table ──────────────────────────────────────────────
phase3_f1_overall = 0.6694
phase3_f1_seca    = 0.0000
phase3_recall     = 0.7417
phase3_precision  = 0.6100

print('=' * 60)
print('COMPARISON: Phase 3 (1-stage) vs Phase 4 (2-stage)')
print('=' * 60)
print(f'{"Metric":<20} {"Phase3 (1-stage)":>18} {"Phase4 (2-stage)":>18}')
print('-' * 60)
print(f'{"F1 Overall":<20} {phase3_f1_overall:>18.4f} {f1_overall_test:>18.4f}')
print(f'{"F1 Healthy":<20} {"—":>18} {f1_healthy_test:>18.4f}')
print(f'{"F1 Seca (KEY)":<20} {phase3_f1_seca:>18.4f} {f1_seca_test:>18.4f}')
print(f'{"Precision":<20} {phase3_precision:>18.4f} {"(crop level)":>18}')
print(f'{"Recall":<20} {phase3_recall:>18.4f} {"(crop level)":>18}')
print('=' * 60)

improvement = f1_seca_test - phase3_f1_seca
if improvement > 0:
    print(f'\n✓ F1-Seca improved by +{improvement:.4f} ({improvement*100:.1f} pp)')
else:
    print(f'\n✗ F1-Seca did not improve ({improvement:.4f} pp). Needs more Seca annotations.')

In [ ]:
# ── Cell 13: Save config and final summary ─────────────────────────────────
config = {
    'architecture':       'Two-Stage: DeepForest detector + ResNet-18 classifier',
    'stage1':             'weecology/deepforest-tree (pretrained, 1-class)',
    'stage2':             'ResNet-18 (ImageNet pretrained, 2-class head)',
    'crop_size':          CROP_SIZE,
    'crop_padding':       PADDING,
    'epochs_trained':     len(train_losses),
    'warmup_epochs':      WARMUP_EPOCHS,
    'learning_rate':      LR,
    'batch_size':         BATCH_SIZE,
    'weighted_sampler':   True,
    'weighted_loss':      True,
    'device':             DEVICE,
    'seed':               SEED,
    'f1_seca_val_best':   float(best_f1_seca),
    'f1_seca_test':       float(f1_seca_test),
    'f1_healthy_test':    float(f1_healthy_test),
    'f1_overall_test':    float(f1_overall_test),
    'phase3_f1_seca':     phase3_f1_seca,
    'improvement_seca':   float(f1_seca_test - phase3_f1_seca),
}
with open(MODELS_DIR / 'stage2_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Config saved: models/stage2_config.json')

# ── Final summary ──────────────────────────────────────────────────────────
print()
print('╔══════════════════════════════════════════════════════════════╗')
print('║         PHASE 4 — TWO-STAGE RESULTS SUMMARY                 ║')
print('╚══════════════════════════════════════════════════════════════╝')
print(f'Stage 2 Classifier (ResNet-18 on crops):')
print(f'  F1 Overall  : {f1_overall_test:.4f}')
print(f'  F1 Healthy  : {f1_healthy_test:.4f}')
print(f'  F1 Seca     : {f1_seca_test:.4f}  (Phase 3 was: 0.0000)')
print(f'  Best val F1-Seca during training: {best_f1_seca:.4f}')
print()
print(f'Saved artifacts:')
print(f'  models/stage2_classifier.pt')
print(f'  models/stage2_config.json')
print(f'  reports/stage2_training_curves.png')
print(f'  reports/stage2_confusion_matrix.png')
print(f'  reports/pipeline_result_[1-3].png')
print(f'  reports/sample_crops.png')
print('══════════════════════════════════════════════════════════════')